# 08 · Inversiones, integración y decisión

Cierra el proyecto con valoración de inversiones, prioridades y un paquete de KPIs para el dashboard ejecutivo.

> **Fuente:** datos sintéticos de Levante Ferries. Proyecto demostrativo; no contiene información real de ninguna naviera.

## Contexto y método

El notebook forma parte de una cadena reproducible. Las fórmulas y supuestos se muestran junto a los resultados para que cada conclusión pueda revisarse.

In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
for folder in [PROCESSED, TABLES, FIGURES]: folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#1f6feb','#2dd4bf','#f59e0b','#ef4444','#94a3b8'])
plt.rcParams.update({'figure.figsize': (11, 5.5), 'axes.titlesize': 14, 'axes.labelsize': 10})
pd.options.display.float_format = '{:,.2f}'.format


## Evaluación de inversiones

In [2]:
rate=.085
initiatives=[('Optimización energética de flota',6_200_000,1_380_000,8,1.1),('Revenue management avanzado',1_450_000,610_000,5,.4),('Electrificación en puerto',4_800_000,880_000,10,1.7)]
rows=[]
for name,capex,annual_cf,life,risk in initiatives:
 cash=[-capex]+[annual_cf]*life
 npv=sum(cf/((1+rate)**t) for t,cf in enumerate(cash))
 try:
  import numpy_financial as npf
  irr=npf.irr(cash)
 except Exception:
  irr=np.nan
 payback=capex/annual_cf
 rows.append({'initiative':name,'capex':capex,'annual_cashflow':annual_cf,'life_years':life,'npv':npv,'irr':irr,'payback_years':payback,'risk_score':risk})
inv=pd.DataFrame(rows).sort_values('npv',ascending=False)
inv['decision']=np.where((inv.npv>0)&(inv.payback_years<6),'Priorizar','Fasear / revisar')
inv.to_csv(TABLES/'08_investment_cases.csv',index=False)
display(inv)

                         initiative    capex  ...  risk_score   decision
0  Optimización energética de flota  6200000  ...        1.10  Priorizar
2         Electrificación en puerto  4800000  ...        1.70  Priorizar
1       Revenue management avanzado  1450000  ...        0.40  Priorizar

[3 rows x 9 columns]


## Mapa de decisión

In [3]:
a=pd.read_csv(RAW/'fact_finance_actual.csv',parse_dates=['month'])
r=pd.read_csv(TABLES/'03_route_profitability.csv')
r['priority']=np.select([r.ebitda_margin<.08,r.safety_margin_pp<8,r.ebitda_margin>.15],['Reestructurar','Proteger margen','Escalar / defender'],default='Optimizar')
r[['route_id','revenue','ebitda','ebitda_margin','safety_margin_pp','priority']].to_csv(TABLES/'08_route_decision_map.csv',index=False)
display(r[['route_id','ebitda_margin','safety_margin_pp','priority']])

  route_id  ebitda_margin  safety_margin_pp            priority
0  DEN-PMI           0.18              8.87  Escalar / defender
1  DEN-FOR           0.08              4.47     Proteger margen
2  VAL-IBZ           0.06              1.80       Reestructurar
3  DEN-IBZ           0.06              2.69       Reestructurar
4  BCN-PMI           0.05              1.08       Reestructurar
5  VAL-PMI           0.05              0.95       Reestructurar
6  BCN-IBZ           0.03              0.07       Reestructurar


## Paquete ejecutivo

In [4]:
cash=pd.read_csv(TABLES/'06_cashflow_annual.csv')
sc=pd.read_csv(TABLES/'07_scenarios_2026.csv')
summary={'period':'2024–2026','revenue_total':float(a.revenue.sum()),'ebitda_total':float(a.ebitda.sum()),'ebitda_margin':float(a.ebitda.sum()/a.revenue.sum()),'free_cash_flow_total':float(cash.free_cash_flow.sum()),'cash_conversion':float(cash.operating_cash_flow.sum()/cash.ebitda.sum()),'best_route':str(r.sort_values('ebitda_margin',ascending=False).iloc[0].route_id),'route_at_risk':str(r.sort_values('safety_margin_pp').iloc[0].route_id),'fuel_shock_impact':float(sc.loc[sc.scenario.eq('Fuel +15%'),'delta_vs_base'].iloc[0]),'top_investment':str(inv.iloc[0].initiative)}
(TABLES/'08_executive_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
summary

## Conclusiones

Las conclusiones concretas se generan a partir de las salidas ejecutadas. Deben interpretarse como evidencia de una simulación y como demostración del método analítico.